### **Manipulación de JSON con Redis**

*Autor: mCárdenas 2026*

#### **1. Instalación de Redis y RedisJSON**

1. Asegúrate de tener Docker instalado. Si no lo tienes, puedes instalarlo siguiendo [estas instrucciones](https://docs.docker.com/get-docker/).

2. Ejecuta un contenedor de Redis con soporte para RedisJSON:
   ```bash
   docker run -d --name redis-json -p 6379:6379 redis/redis-stack:latest
   ```

   Este contenedor incluye la extensión **RedisJSON**.

3. Instala la librería `redis` en Python:
   ```bash
   pip install redis
   ```

#### **2. Conceptos clave**

- RedisJSON es un módulo de Redis que permite almacenar, recuperar y manipular documentos JSON directamente en Redis.
- Usa comandos como `JSON.SET`, `JSON.GET`, `JSON.ARRAPPEND`, etc., para interactuar con datos JSON.

#### **3. Estructura del taller**

El taller se divide en las siguientes partes:
1. Conexión a Redis.
2. Almacenar un documento JSON.
3. Recuperar datos JSON.
4. Actualizar datos JSON.
5. Manipulación avanzada: arrays y objetos anidados.

### **Código del taller**

In [ ]:
import redis
import json

# Conexión al servidor Redis
redis_client = redis.StrictRedis(host='localhost', port=6379, decode_responses=True)

# Función para mostrar resultados del taller
def print_title(title):
    print(f"\n{'=' * 20} {title} {'=' * 20}\n")

# 1. Almacenar un documento JSON en Redis
print_title("1. Almacenar un documento JSON")
document = {
    "id": 1,
    "nombre": "Carlos Pérez",
    "direccion": {
        "calle": "Calle 50",
        "ciudad": "Monterrey",
        "codigoPostal": "64000"
    },
    "email": "carlos.perez@example.com",
    "puntosFidelidad": 500,
    "historialCompras": [
        {"producto": "Laptop", "precio": 1200},
        {"producto": "Teclado", "precio": 50}
    ]
}

In [ ]:
# Almacenar el JSON en Redis
redis_client.execute_command("JSON.SET", "cliente:1", ".", json.dumps(document))
print("Documento almacenado con éxito.")

In [ ]:
# 2. Recuperar datos JSON desde Redis
print_title("2. Recuperar datos JSON")
retrieved_document = redis_client.execute_command("JSON.GET", "cliente:1")
print("Documento recuperado:", json.loads(retrieved_document))

In [ ]:
# 3. Actualizar datos JSON en Redis
print_title("3. Actualizar datos JSON")
# Incrementar puntos de fidelidad
redis_client.execute_command("JSON.NUMINCRBY", "cliente:1", ".puntosFidelidad", 100)
updated_points = redis_client.execute_command("JSON.GET", "cliente:1", ".puntosFidelidad")
print("Puntos de fidelidad actualizados:", updated_points)

In [ ]:
# Cambiar la ciudad de la dirección
redis_client.execute_command("JSON.SET", "cliente:1", ".direccion.ciudad", '"Guadalajara"')
updated_city = redis_client.execute_command("JSON.GET", "cliente:1", ".direccion.ciudad")
print("Ciudad actualizada:", updated_city)

In [ ]:
# 4. Manipular arrays en JSON
print_title("4. Manipular arrays en JSON")
# Agregar una nueva compra al historial
redis_client.execute_command("JSON.ARRAPPEND", "cliente:1", ".historialCompras", json.dumps({"producto": "Mouse", "precio": 20}))
updated_history = redis_client.execute_command("JSON.GET", "cliente:1", ".historialCompras")
print("Historial de compras actualizado:", json.loads(updated_history))

In [ ]:
# Eliminar el primer elemento del historial
redis_client.execute_command("JSON.ARRPOP", "cliente:1", ".historialCompras", 0)
updated_history_after_pop = redis_client.execute_command("JSON.GET", "cliente:1", ".historialCompras")
print("Historial de compras después de eliminar el primer elemento:", json.loads(updated_history_after_pop))

In [ ]:
# 5. Operaciones avanzadas con JSON
print_title("5. Operaciones avanzadas con JSON")
# Verificar la existencia de una clave
key_exists = redis_client.execute_command("JSON.GET", "cliente:1", ".direccion.ciudad")
if key_exists:
    print("La clave '.direccion.ciudad' existe:", json.loads(key_exists))
else:
    print("La clave '.direccion.ciudad' no existe.")

In [ ]:
# Eliminar un campo específico
redis_client.execute_command("JSON.DEL", "cliente:1", ".email")
updated_document = redis_client.execute_command("JSON.GET", "cliente:1")
print("Documento después de eliminar el campo 'email':", json.loads(updated_document))

### **6. Ejecución paso a paso**

#### Paso 1: Almacenar un documento JSON
El JSON se almacena en Redis con el comando `JSON.SET`. Esto crea una estructura de datos JSON directamente en Redis.

#### Paso 2: Recuperar datos
Usamos `JSON.GET` para recuperar el JSON completo o partes específicas.

#### Paso 3: Actualizar datos
Actualizamos campos específicos usando:
- `JSON.NUMINCRBY` para incrementar valores numéricos.
- `JSON.SET` para modificar directamente cualquier clave.

#### Paso 4: Manipular arrays
- Agregamos elementos con `JSON.ARRAPPEND`.
- Eliminamos elementos de un array con `JSON.ARRPOP`.

#### Paso 5: Operaciones avanzadas
- Verificamos la existencia de una clave.
- Eliminamos claves específicas del JSON con `JSON.DEL`.


### **Salida esperada**

```plaintext
==================== 1. Almacenar un documento JSON ====================

Documento almacenado con éxito.

==================== 2. Recuperar datos JSON ====================

Documento recuperado: {'id': 1, 'nombre': 'Carlos Pérez', ...}

==================== 3. Actualizar datos JSON ====================

Puntos de fidelidad actualizados: 600
Ciudad actualizada: Guadalajara

==================== 4. Manipular arrays en JSON ====================

Historial de compras actualizado: [{'producto': 'Laptop', 'precio': 1200}, ...]
Historial de compras después de eliminar el primer elemento: [{'producto': 'Teclado', 'precio': 50}, ...]

==================== 5. Operaciones avanzadas con JSON ====================

La clave '.direccion.ciudad' existe: Guadalajara
Documento después de eliminar el campo 'email': {'id': 1, 'nombre': 'Carlos Pérez', ...}
```

### **Notas adicionales**

- RedisJSON es extremadamente rápido para trabajar con datos JSON estructurados.
- Puedes personalizar el documento JSON según las necesidades de tu aplicación.
- Para verificar que RedisJSON está activo, asegúrate de ejecutar el contenedor con la imagen de Redis Stack.